In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools','opencv-python','Pillow'],check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import torch,shutil,time
ROOT=Path('/content/drive/MyDrive/Vision-IA/detector_D2')
WORK=Path('/content/vision_detector_D2')
assert torch.cuda.is_available(), 'Selecciona GPU T4'
assert ROOT.is_dir()
shutil.copytree(ROOT/'source',WORK,dirs_exist_ok=True)
for name in ['training','scripts']:
    (WORK/name/'__init__.py').write_text('')
print(torch.cuda.get_device_name(0))


In [ ]:
import zipfile,hashlib,json
DATA=WORK/'taco'
DATA.mkdir(parents=True,exist_ok=True)
archive=ROOT.parent/'detector_D1/taco_verified.zip'
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        name=info.filename
        if name.startswith(('images/train/','images/val/')) or name in ['manifest.json','audit.json','annotations.json']:
            assert DATA.resolve() in (DATA/name).resolve().parents
            z.extract(info,DATA)
rows=json.loads((DATA/'manifest.json').read_text())
for row in rows:
    if row['split'] not in ['train','val']:continue
    name=f"{row['id']:05d}.jpg"
    backup=ROOT/'verified_originals'/name
    target=DATA/'images'/row['split']/name
    if backup.exists():shutil.copy2(backup,target)
    assert hashlib.sha256(target.read_bytes()).hexdigest()==row['sha256'], target
print('Train/val verificados; test reservado en el archivo')

# Additional human-annotated MJU train images; original TACO holdouts unchanged.
import os
MIXED=WORK/'mixed_D2';MIXED.mkdir(exist_ok=True)
with zipfile.ZipFile(ROOT/'packages/mju_train_append.zip') as z:
    for info in z.infolist():
        assert (MIXED/info.filename).resolve().is_relative_to(MIXED.resolve())
    z.extractall(MIXED)
for row in json.loads((MIXED/'manifest.json').read_text()):
    if row['split']=='test':continue
    relative=Path('images')/row['split']/f"{row['id']:05d}.jpg"
    target=MIXED/relative
    if not target.exists():
        target.parent.mkdir(parents=True,exist_ok=True)
        os.link(DATA/relative,target)
    assert hashlib.sha256(target.read_bytes()).hexdigest()==row['sha256']
DATA=MIXED
print('Train: 1706, validation: 197. Test not extracted.')


In [ ]:
SMOKE=ROOT/'runs'/('smoke_'+time.strftime('%Y%m%d_%H%M%S',time.gmtime()))
command=[sys.executable,'-u','-m','scripts.train_detector_own','--config',str(WORK/'config/detector_D2.json'),'--root',str(DATA),'--output',str(SMOKE),'--smoke']
subprocess.run(command,cwd=WORK,check=True)


In [ ]:
# Reanudar solo un last.pth propio de D2. Vacio crea pesos nuevos.
RESUME=''
RUN=Path(RESUME).parent if RESUME else ROOT/'runs'/('D2_'+time.strftime('%Y%m%d_%H%M%S',time.gmtime()))
command=[sys.executable,'-u','-m','scripts.train_detector_own','--config',str(WORK/'config/detector_D2.json'),'--root',str(DATA),'--output',str(RUN)]
if RESUME:command+=['--resume',RESUME]
subprocess.run(command,cwd=WORK,check=True)
